In [3]:
import pandas as pd

In [ ]:
#read
df=pd.read_csv("insurance_claims.csv", parse_dates=["policy_bind_date", "incident_date"], index_col="policy_number")

#check duplicates
df.duplicated().sum() 
df['authorities_contacted']=df['authorities_contacted'].fillna("Police")

#ensure consistency on column names
rename ={'capital-gains':'capital_gains','capital-loss': 'capital_loss'}
df.rename(columns=rename, inplace=True)

#Rename Strings
df["collision_type"]=df["collision_type"].apply(lambda x: "No Collision" if str(x)=="?" else x)
df["police_report_available"]=df["police_report_available"].apply(lambda x: "Unknown" if str(x)=="?" else x)
df["property_damage"]=df["property_damage"].apply(lambda x: "Unknown" if str(x)=="?" else x)

# Drop

In [5]:
# drop column as 100% nan-values
# drop umbrella limits as 80% of values are zeros, std = 1.1 Mio
df.drop(['months_as_customer',  'umbrella_limit',  'total_claim_amount', '_c39'],axis=1,inplace= True)

In [6]:
df.isna().sum().sum()

np.int64(0)

In [7]:
# #Type df.drop stuff
# for col in df.columns:
#     print(f"df.drop('{col}',axis=1,inplace=True)")

In [8]:
df.drop('age',axis=1,inplace=True)
df.drop('policy_bind_date',axis=1,inplace=True)
df.drop('policy_state',axis=1,inplace=True)
df.drop('policy_csl',axis=1,inplace=True)
df.drop('policy_deductable',axis=1,inplace=True)
df.drop('policy_annual_premium',axis=1,inplace=True)
df.drop('insured_zip',axis=1,inplace=True)
df.drop('insured_sex',axis=1,inplace=True)
df.drop('insured_education_level',axis=1,inplace=True)
df.drop('insured_occupation',axis=1,inplace=True)
df.drop('insured_relationship',axis=1,inplace=True)
df.drop('capital_gains',axis=1,inplace=True)
df.drop('capital_loss',axis=1,inplace=True)
df.drop('incident_date',axis=1,inplace=True)
df.drop('incident_city',axis=1,inplace=True)
df.drop('incident_location',axis=1,inplace=True)
df.drop('incident_hour_of_the_day',axis=1,inplace=True)
df.drop('number_of_vehicles_involved',axis=1,inplace=True)
df.drop('property_damage',axis=1,inplace=True)
df.drop('bodily_injuries',axis=1,inplace=True)
df.drop('witnesses',axis=1,inplace=True)
df.drop('police_report_available',axis=1,inplace=True)
# df.drop('injury_claim',axis=1,inplace=True)
# df.drop('property_claim',axis=1,inplace=True)
# df.drop('vehicle_claim',axis=1,inplace=True)
df.drop('auto_make',axis=1,inplace=True)
df.drop('auto_model',axis=1,inplace=True)
df.drop('auto_year',axis=1,inplace=True)


In [9]:
df.columns

Index(['insured_hobbies', 'incident_type', 'collision_type',
       'incident_severity', 'authorities_contacted', 'incident_state',
       'injury_claim', 'property_claim', 'vehicle_claim', 'fraud_reported'],
      dtype='object')

In [11]:
from sklearn.preprocessing import LabelEncoder
LabelEnc = LabelEncoder()
df['fraud_reported'] = LabelEnc.fit_transform(df['fraud_reported'])

# Train Test Split

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('fraud_reported', axis=1), df['fraud_reported'], test_size=0.2, random_state=42)

# Encoding


In [13]:
from sklearn.preprocessing import OneHotEncoder

enc=OneHotEncoder(handle_unknown='ignore')
X_train_enc=enc.fit_transform(X_train[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
#dataframe
X_train_enc=pd.DataFrame(X_train_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))

#reset index
X_train_enc.reset_index(drop=True, inplace=True)
X_train.reset_index(drop=True, inplace=True)

#concatenate
X_train=pd.concat([X_train, X_train_enc], axis=1)

#drop columns
X_train.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)

# same for test data
X_test_enc=enc.transform(X_test[['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']]).toarray()
X_test_enc=pd.DataFrame(X_test_enc, columns=enc.get_feature_names_out(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state']))
X_test_enc.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
X_test=pd.concat([X_test, X_test_enc], axis=1)
X_test.drop(['insured_hobbies', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state'], axis=1, inplace=True)


In [15]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

minmax_scaler = MinMaxScaler()
rbst_scaler = RobustScaler()

minmax_col = [ "injury_claim",'vehicle_claim']
rbst_col = ['property_claim']


for column in minmax_col:
    X_train[column] = minmax_scaler.fit_transform(X_train[[column]])
    X_test[column] = minmax_scaler.fit_transform(X_test[[column]])
    
for column in rbst_col:
    X_train[column] = rbst_scaler.fit_transform(X_train[[column]])
    X_test[column] = rbst_scaler.transform(X_test[[column]])
    

In [16]:
#df.to_csv('insurance_claims_clean.csv')
X_train.to_csv("X_train_1.csv")
X_test.to_csv("X_test_1.csv")
y_train.to_csv('y_train_1.csv')
y_test.to_csv('y_test_1.csv')

In [17]:
import pandas as pd
import numpy as np

from scipy.stats import spearmanr, pearsonr,chi2_contingency
from sklearn import model_selection, preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler,  ClusterCentroids
from lazypredict.Supervised import LazyClassifier
from imblearn.metrics import classification_report_imbalanced, geometric_mean_score

In [18]:
ros=RandomOverSampler()
smote=SMOTE()
cc=ClusterCentroids()
rus=RandomUnderSampler()
st = SMOTETomek(random_state=42)

In [19]:
X_ro,y_ro=ros.fit_resample(X_train, y_train)
X_sm,y_sm=smote.fit_resample(X_train, y_train)
X_cc,y_cc=cc.fit_resample(X_train, y_train)
X_ru, y_ru=rus.fit_resample(X_train, y_train)
X_st,y_st = st.fit_resample(X_train, y_train)
print('Oversampled sample classes:', dict(pd.Series(y_ro).value_counts()))
print('SMOTE sample classes:', dict(pd.Series(y_sm).value_counts()))
print('SMOTE sample classes:', dict(pd.Series(y_cc).value_counts()))
print(dict(pd.Series(y_ru).value_counts()))

Oversampled sample classes: {0: np.int64(608), 1: np.int64(608)}
SMOTE sample classes: {0: np.int64(608), 1: np.int64(608)}
SMOTE sample classes: {0: np.int64(192), 1: np.int64(192)}
{0: np.int64(192), 1: np.int64(192)}


In [20]:
X_list=(X_train,X_ro,X_sm,X_cc,X_ru,X_st)
y_list=(y_train,y_ro,y_sm, y_cc,y_ru,y_st)
sample =('Original','Ro', 'SMOTE', 'CC', 'ST')

for u,v,sample_name in zip(X_list,y_list,sample):
    clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
    models,predictions = clf.fit(u, X_test, v, y_test)
    print(sample_name)
    display(models)
    

100%|██████████| 32/32 [00:02<00:00, 12.49it/s]

[LightGBM] [Info] Number of positive: 192, number of negative: 608
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 894
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.240000 -> initscore=-1.152680
[LightGBM] [Info] Start training from score -1.152680
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LinearDiscriminantAnalysis,0.85,0.85,0.85,0.85,0.05
SVC,0.84,0.82,0.82,0.85,0.08
LinearSVC,0.83,0.81,0.81,0.84,0.03
RidgeClassifierCV,0.83,0.81,0.81,0.83,0.04
RidgeClassifier,0.83,0.81,0.81,0.83,0.04
NearestCentroid,0.81,0.81,0.81,0.81,0.02
ExtraTreesClassifier,0.84,0.81,0.81,0.85,0.29
BernoulliNB,0.81,0.80,0.80,0.82,0.03
SGDClassifier,0.81,0.79,0.79,0.81,0.03


100%|██████████| 32/32 [00:03<00:00,  9.61it/s]


[LightGBM] [Info] Number of positive: 608, number of negative: 608
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000939 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 889
[LightGBM] [Info] Number of data points in the train set: 1216, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Ro


...

[LightGBM] [Info] Number of positive: 192, number of negative: 192
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000602 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 666
[LightGBM] [Info] Number of data points in the train set: 384, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
RidgeClassifier,0.84,0.84,0.84,0.84,0.03
RidgeClassifierCV,0.84,0.84,0.84,0.84,0.04
LinearDiscriminantAnalysis,0.84,0.84,0.84,0.84,0.04
LinearSVC,0.83,0.84,0.84,0.84,0.03
CalibratedClassifierCV,0.83,0.83,0.83,0.84,0.07
LogisticRegression,0.82,0.82,0.82,0.83,0.02
ExtraTreesClassifier,0.80,0.82,0.82,0.80,0.23
BernoulliNB,0.80,0.82,0.82,0.80,0.03
RandomForestClassifier,0.78,0.81,0.81,0.79,0.26


100%|██████████| 32/32 [00:02<00:00, 14.10it/s]

[LightGBM] [Info] Number of positive: 192, number of negative: 192
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000513 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 472
[LightGBM] [Info] Number of data points in the train set: 384, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
AdaBoostClassifier,0.83,0.85,0.85,0.84,0.20
CalibratedClassifierCV,0.83,0.85,0.85,0.84,0.08
LinearDiscriminantAnalysis,0.83,0.85,0.85,0.84,0.04
RidgeClassifierCV,0.83,0.85,0.85,0.84,0.04
RidgeClassifier,0.83,0.85,0.85,0.84,0.02
LinearSVC,0.83,0.85,0.85,0.84,0.02
LGBMClassifier,0.82,0.84,0.84,0.83,0.10
SVC,0.83,0.84,0.84,0.84,0.05
NuSVC,0.83,0.84,0.84,0.84,0.05


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_ro,y_ro=ros.fit_resample(X_train, y_train)
X_sm,y_sm=smote.fit_resample(X_train, y_train)
X_cc,y_cc=cc.fit_resample(X_train, y_train)
X_ru, y_ru=rus.fit_resample(X_train, y_train)
X_st,y_st = st.fit_resample(X_train, y_train)
print('Oversampled sample classes:', dict(pd.Series(y_ro).value_counts()))
print('SMOTE sample classes:', dict(pd.Series(y_sm).value_counts()))
print('SMOTE sample classes:', dict(pd.Series(y_cc).value_counts()))
print(dict(pd.Series(y_ru).value_counts()))

X_list=(X_train,X_ro,X_sm,X_cc,X_ru,X_st)
y_list=(y_train,y_ro,y_sm, y_cc,y_ru,y_st)
sample =('Original','Ro', 'SMOTE', 'CC', 'ST')

for u,v, sample_name in zip(X_list,y_list, sample):
    lr=LogisticRegression()
    lr.fit(u,v)
    print(sample_name)
    print("Train score:",lr.score(u,v))
    print("Test score: ",lr.score(X_test,y_test))

    print(pd.crosstab(y_test,lr.predict(X_test)))
    print(classification_report(y_test, lr.predict(X_test)))

Oversampled sample classes: {0: np.int64(608), 1: np.int64(608)}
SMOTE sample classes: {0: np.int64(608), 1: np.int64(608)}
SMOTE sample classes: {0: np.int64(192), 1: np.int64(192)}
{0: np.int64(192), 1: np.int64(192)}
Original
Train score: 0.87375
Test score:  0.81
col_0             0   1
fraud_reported         
0               126  19
1                19  36
              precision    recall  f1-score   support

           0       0.87      0.87      0.87       145
           1       0.65      0.65      0.65        55

    accuracy                           0.81       200
   macro avg       0.76      0.76      0.76       200
weighted avg       0.81      0.81      0.81       200

Ro
Train score: 0.8741776315789473
Test score:  0.84
col_0             0   1
fraud_reported         
0               120  25
1                 7  48
              precision    recall  f1-score   support

           0       0.94      0.83      0.88       145
           1       0.66      0.87      0.75        